# PSMamba on Kaggle: Setup and Smoke Test

**Before you run:** in the right sidebar set **Accelerator = GPU T4 x2** (one GPU is enough; avoid P100) and **Internet = On**.

This notebook does 5 things:
1. Checks the GPU, Python, torch and CUDA versions
2. Installs `causal-conv1d` and `mamba-ssm` (the same major version the paper used, 2.2.3)
3. (Optional) saves the built wheels so you never have to install again
4. Smoke-tests Mamba and a small bidirectional Mamba encoder on 224 / 160 / 96 crops
5. Lists your input datasets and defines seed and checkpoint helpers

## 1. Check the environment

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} | {p.total_memory/1e9:.1f} GB | compute capability {p.major}.{p.minor}")
else:
    print("NO GPU. Turn on the accelerator in the sidebar, then restart the session.")

In [ ]:
!nvcc --version | tail -n 2

## 2. Install Mamba

The `mamba-ssm` installer first tries to download a **prebuilt wheel** that matches your torch / CUDA / Python. If none exists it compiles from source, which can take 30+ minutes. Let it run.

If it fails, copy the last 20 lines of the error and send them to me.

In [ ]:
%%time
import os
os.environ["MAX_JOBS"] = "4"   # limits RAM use if it has to compile
!pip install -q ninja packaging
!pip install -q causal-conv1d==1.4.0 --no-build-isolation
!pip install -q mamba-ssm==2.2.3 --no-build-isolation

## 3. (Optional) Save wheels for reuse

Run this once. Then: **Save Version → Save & Run All**, open the finished version's *Output* tab, click **New Dataset**, and name it e.g. `mamba-wheels`.

In later notebooks, add that dataset and install in seconds with:

```
!pip install --no-index --find-links=/kaggle/input/mamba-wheels causal-conv1d mamba-ssm
```

In [ ]:
!mkdir -p /kaggle/working/wheels
!pip wheel -q --no-deps --no-build-isolation -w /kaggle/working/wheels causal-conv1d==1.4.0 mamba-ssm==2.2.3
!ls -lh /kaggle/working/wheels

## 4. Smoke tests

**Test A:** one Mamba layer, forward + backward in fp16 (the T4 has no bf16).

In [ ]:
import torch
from mamba_ssm import Mamba

dev = "cuda"
m = Mamba(d_model=192, d_state=16, d_conv=4, expand=2).to(dev)
x = torch.randn(8, 196, 192, device=dev, requires_grad=True)
with torch.autocast("cuda", dtype=torch.float16):
    y = m(x)
loss = y.float().pow(2).mean()
loss.backward()
print("Test A OK | out:", tuple(y.shape), "| loss:", round(loss.item(), 4),
      "| grad finite:", torch.isfinite(x.grad).all().item())

**Test B:** a small *bidirectional* Mamba encoder (simplified from the paper's Eq. 7: forward scan + backward scan, fused by learnable weights).

Note: this is a minimal stand-in, **not** the official Vision Mamba code. It is enough to confirm that multi-crop inputs (224 / 160 / 96) work, to see memory and speed, and to build on later. With patch size 16 the token grids are 14x14, 10x10 and 6x6.

In [ ]:
import time, torch, torch.nn as nn, torch.nn.functional as F
from mamba_ssm import Mamba

class BiMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.fwd = Mamba(d_model=d, d_state=16, d_conv=4, expand=2)
        self.bwd = Mamba(d_model=d, d_state=16, d_conv=4, expand=2)
        self.wf = nn.Parameter(torch.tensor(0.5))   # learnable fusion weights (paper Eq. 7)
        self.wb = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        h = self.norm(x)
        f = self.fwd(h)
        b = self.bwd(h.flip(1)).flip(1)
        return x + self.wf * f + self.wb * b

class MiniBiMamba(nn.Module):
    def __init__(self, d=192, depth=6, patch=16, base_grid=14):
        super().__init__()
        self.patch = patch
        self.base_grid = base_grid
        self.embed = nn.Conv2d(3, d, patch, patch)
        self.pos = nn.Parameter(torch.zeros(1, d, base_grid, base_grid))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([BiMambaBlock(d) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
    def forward(self, img):
        t = self.embed(img)                                   # B, d, gh, gw
        gh, gw = t.shape[-2:]
        pos = F.interpolate(self.pos, size=(gh, gw), mode="bicubic", align_corners=False)
        t = (t + pos).flatten(2).transpose(1, 2)              # B, N, d
        for blk in self.blocks:
            t = blk(t)
        return self.norm(t), (gh, gw)

model = MiniBiMamba().to("cuda")
print("Params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))

for size in (224, 160, 96):
    torch.cuda.reset_peak_memory_stats()
    x = torch.randn(8, 3, size, size, device="cuda")
    torch.cuda.synchronize(); t0 = time.time()
    with torch.autocast("cuda", dtype=torch.float16):
        tokens, grid = model(x)
    tokens.float().pow(2).mean().backward()
    torch.cuda.synchronize()
    print(f"{size}x{size}: grid {grid}, tokens {tuple(tokens.shape)}, "
          f"{time.time()-t0:.2f}s, peak mem {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print("Test B OK")

## 5. Helpers: seed and checkpoint/resume

Kaggle sessions can die, so save often and resume from `/kaggle/working`.

In [ ]:
import os, random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

CKPT = "/kaggle/working/ckpt.pt"

def save_ckpt(state, path=CKPT):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)          # atomic, so a crash never leaves a half-written file

def load_ckpt(path=CKPT):
    return torch.load(path, map_location="cpu") if os.path.exists(path) else None

set_seed(42)
print("helpers ready | existing checkpoint:", os.path.exists(CKPT))

## 6. Find your datasets

Add PlantDoc and Citrus via **Add Input** (search "PlantDoc", "citrus leaves"). They appear under `/kaggle/input`.

In [ ]:
import os
root = "/kaggle/input"
for name in sorted(os.listdir(root)):
    p = os.path.join(root, name)
    n = sum(len(f) for _, _, f in os.walk(p))
    print(f"{name}: {n} files")
    for sub in sorted(os.listdir(p))[:8]:
        print("   -", sub)

## Done: what to check

- Test A and Test B both print **OK**
- Peak memory for batch 8 at 224 px is comfortably below 15 GB
- Your datasets are listed

Next notebook: load PlantDoc / Citrus, then a fine-tuning baseline.